# EXP05: 时段择时全套分析

**合并**: NB10 + NB11 + NB12 + NB13

1. 按入场小时 PnL 分解
2. 4 时段分区深度统计 (按年份交叉)
3. 跳过亏损时段模拟
4. K-Fold 稳定性验证

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Baseline: {len(trades)} trades, PnL=${sum(t.pnl for t in trades):.0f}")

In [ ]:
def session(h):
    if 0 <= h < 8: return 'Asia(0-7)'
    if 8 <= h < 14: return 'London(8-13)'
    if 14 <= h < 21: return 'NY(14-20)'
    return 'Late(21-23)'

df = pd.DataFrame([{
    'year': t.entry_time.year,
    'hour': t.entry_time.hour,
    'session': session(t.entry_time.hour),
    'pnl': t.pnl,
    'strategy': t.strategy,
    'direction': t.direction,
    'exit_reason': t.exit_reason.split(':')[0] if ':' in t.exit_reason else t.exit_reason,
    'bars_held': t.bars_held,
    'win': 1 if t.pnl > 0 else 0,
} for t in trades])

## Part 1: 按入场小时 PnL 分解

In [ ]:
hourly = df.groupby('hour').agg(
    count=('pnl', 'count'), total_pnl=('pnl', 'sum'), avg_pnl=('pnl', 'mean'),
    win_rate=('win', 'mean'), avg_bars=('bars_held', 'mean'),
).round(2)
print("=== PnL by Entry Hour (UTC) ===")
print(hourly)
print(f"\nBest hour: {hourly['avg_pnl'].idxmax()} (${hourly['avg_pnl'].max():.2f}/trade)")
print(f"Worst hour: {hourly['avg_pnl'].idxmin()} (${hourly['avg_pnl'].min():.2f}/trade)")

## Part 2: 4 时段分区深度统计

In [ ]:
sess = df.groupby('session').agg(
    count=('pnl', 'count'), total_pnl=('pnl', 'sum'), avg_pnl=('pnl', 'mean'), win_rate=('win', 'mean'),
).round(2)
print("=== PnL by Session ===")
print(sess)

print("\n=== PnL by Session x Strategy ===")
print(df.groupby(['session', 'strategy'])['pnl'].agg(['sum', 'mean', 'count']).round(2))

print("\n=== PnL by Session x Direction ===")
print(df.groupby(['session', 'direction'])['pnl'].agg(['sum', 'mean', 'count']).round(2))

In [ ]:
# 按年份 x 时段交叉
pivot = df.pivot_table(index='year', columns='session', values='pnl', aggfunc='sum').round(0)
print("=== Total PnL by Year x Session ===")
print(pivot)

print("\n=== Avg PnL/trade by Year x Session ===")
print(df.pivot_table(index='year', columns='session', values='pnl', aggfunc='mean').round(2))

# 时段稳定性
print("\n=== Session Consistency (years with positive PnL) ===")
for sess_name in ['Asia(0-7)', 'London(8-13)', 'NY(14-20)', 'Late(21-23)']:
    col = pivot.get(sess_name)
    if col is not None:
        pos = (col > 0).sum()
        total = len(col.dropna())
        print(f"  {sess_name}: {pos}/{total} years positive, total ${col.sum():.0f}")

## Part 3: 跳过亏损时段模拟

In [ ]:
def filter_by_hours(trades, allowed_hours, label):
    filtered = [t for t in trades if t.entry_time.hour in allowed_hours]
    total_pnl = sum(t.pnl for t in filtered)
    n = len(filtered)
    wins = sum(1 for t in filtered if t.pnl > 0)
    wr = wins / n * 100 if n > 0 else 0
    avg_pnl = total_pnl / n if n > 0 else 0
    print(f"  {label:30s}: N={n:5d}  PnL=${total_pnl:8.0f}  $/trade={avg_pnl:6.2f}  WR={wr:5.1f}%")
    return {'label': label, 'n': n, 'total_pnl': total_pnl, 'avg_pnl': avg_pnl, 'wr': wr}

print("=== Session Filter Scenarios ===")
scenarios = [
    (list(range(24)), "All hours (baseline)"),
    (list(range(8, 24)), "Skip Asia (8-23)"),
    (list(range(14, 21)), "NY only (14-20)"),
    (list(range(14, 24)), "NY+Late (14-23)"),
    (list(range(0, 8)) + list(range(14, 24)), "Skip London (0-7,14-23)"),
]
for hours, label in scenarios:
    filter_by_hours(trades, hours, label)

print("\n*** 注意: 后处理过滤是上限估计，实际效果可能更小 ***")

## Part 4: K-Fold 时段稳定性验证

In [ ]:
folds = [
    ("Fold1", "2015-01-01", "2017-01-01"),
    ("Fold2", "2017-01-01", "2019-01-01"),
    ("Fold3", "2019-01-01", "2021-01-01"),
    ("Fold4", "2021-01-01", "2023-01-01"),
    ("Fold5", "2023-01-01", "2025-01-01"),
    ("Fold6", "2025-01-01", "2026-04-01"),
]

# 根据 Part 3 中最有价值的过滤方案填入
BEST_HOURS = list(range(8, 24))
FILTER_LABEL = "Skip_Asia"

fold_results = []
for fold_name, start, end in folds:
    fold_data = data.slice(start, end)
    if len(fold_data.m15_df) < 1000:
        continue
    r = run_variant(fold_data, f"{fold_name}", **LIVE_KWARGS)
    all_pnl = sum(t.pnl for t in r['_trades'])
    filt_trades = [t for t in r['_trades'] if t.entry_time.hour in BEST_HOURS]
    filt_pnl = sum(t.pnl for t in filt_trades)
    fold_results.append({
        'fold': fold_name, 'n_all': len(r['_trades']), 'pnl_all': all_pnl,
        'n_filt': len(filt_trades), 'pnl_filt': filt_pnl, 'delta': filt_pnl - all_pnl,
    })

rdf = pd.DataFrame(fold_results)
print(f"\n=== K-Fold: {FILTER_LABEL} ===")
print(rdf.to_string(index=False))
wins = (rdf['delta'] > 0).sum()
print(f"\nFilter wins {wins}/{len(rdf)} folds")